<a href="https://colab.research.google.com/github/pozdnyavladimer-jpg/geometric-state-navigator/blob/main/Untitled38.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!rm -rf /content/geometric-state-navigator /content/gitcube-lab

!git clone https://github.com/pozdnyavladimer-jpg/geometric-state-navigator.git
!git clone https://github.com/pozdnyavladimer-jpg/gitcube-lab.git

Cloning into 'geometric-state-navigator'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 125 (delta 58), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 1.19 MiB | 4.49 MiB/s, done.
Resolving deltas: 100% (58/58), done.
Cloning into 'gitcube-lab'...
remote: Enumerating objects: 1013, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 1013 (delta 121), reused 77 (delta 77), pack-reused 847 (from 2)
Receiving objects: 100% (1013/1013), 677.18 KiB | 2.88 MiB/s, done.
Resolving deltas: 100% (465/465), done.


In [ ]:
import sys
sys.path.insert(0, "/content/geometric-state-navigator")

from kernel.gitcube_bridge import run_gitcube_risk

import random
from collections import defaultdict, deque
from copy import deepcopy
from pprint import pprint

In [ ]:

def clone_graph(graph):
    return deepcopy(graph)

def edge_set(graph):
    return {(e["from"], e["to"]) for e in graph["edges"]}

def build_adj(graph):
    adj = defaultdict(list)
    for e in graph["edges"]:
        adj[e["from"]].append(e["to"])
    return adj

def shortest_path_length(graph, src, dst):
    if src == dst:
        return 0
    adj = build_adj(graph)
    q = deque([(src, 0)])
    seen = {src}
    while q:
        u, d = q.popleft()
        for v in adj[u]:
            if v == dst:
                return d + 1
            if v not in seen:
                seen.add(v)
                q.append((v, d + 1))
    return None

def has_path(graph, src, dst):
    return shortest_path_length(graph, src, dst) is not None

def count_two_cycles(graph):
    pairs = edge_set(graph)
    count = 0
    seen = set()
    for a, b in pairs:
        if (b, a) in pairs and (b, a) not in seen:
            count += 1
            seen.add((a, b))
            seen.add((b, a))
    return count

In [ ]:

def connectivity_score(graph):
    nodes = graph["nodes"]
    if len(nodes) <= 1:
        return 1.0

    adj = build_adj(graph)
    total = 0
    reachable = 0

    for a in nodes:
        q = deque([a])
        seen = {a}
        while q:
            u = q.popleft()
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    q.append(v)

        for b in nodes:
            if a == b:
                continue
            total += 1
            if b in seen:
                reachable += 1

    return reachable / total if total > 0 else 0.0

def tracked_transfer_score(graph):
    nodes = set(graph["nodes"])
    pairs = []

    if "API" in nodes and "DB" in nodes:
        pairs.append(("API", "DB"))
    if "Service" in nodes and "DB" in nodes:
        pairs.append(("Service", "DB"))
    if "Cache" in nodes and "Service" in nodes:
        pairs.append(("Cache", "Service"))

    if not pairs:
        return 0.0

    total = 0.0
    for src, dst in pairs:
        dist = shortest_path_length(graph, src, dst)
        if dist is None:
            total += 0.0
        else:
            total += 1.0 / (1.0 + dist)

    return total / len(pairs)

def utility_score(graph):
    return 0.5 * connectivity_score(graph) + 0.5 * tracked_transfer_score(graph)

def evaluate_needs(graph, risk_info):
    risk = risk_info["risk"]
    utility = utility_score(graph)
    transfer = tracked_transfer_score(graph)
    violations = risk_info.get("layer_violations", 0)

    return {
        "energy": transfer,
        "stability": max(0.0, 1.0 - risk),
        "repair": max(0.0, 1.0 - min(1.0, violations / 3.0)),
        "growth": utility,
    }

def needs_score(needs):
    return (
        needs["energy"] * 0.30 +
        needs["stability"] * 0.30 +
        needs["repair"] * 0.20 +
        needs["growth"] * 0.20
    )

In [ ]:

def left_hemisphere_propose(graph):
    """
    Ліва півкуля:
    - aggressively prefers lower risk / lower complexity
    - can simplify graph if it stays functional enough
    """
    g = clone_graph(graph)
    base_info = run_gitcube_risk(g, "/content/gitcube-lab")
    base_util = utility_score(g)

    # 1. fix violations first
    if g.get("layer_violations", 0) > 0:
        g["layer_violations"] -= 1
        return g, "left:fix_layer"

    # 2. break 2-cycle
    pairs = edge_set(g)
    for a, b in list(pairs):
        if (b, a) in pairs:
            for i, e in enumerate(g["edges"]):
                if e["from"] == b and e["to"] == a:
                    g["edges"].pop(i)
                    return g, f"left:break_2cycle {b}->{a}"

    # 3. remove one edge if it reduces risk and does not kill utility too much
    best_graph = g
    best_action = "left:none"
    best_gain = 0.0

    for idx, e in enumerate(list(g["edges"])):
        test = clone_graph(g)
        removed = test["edges"].pop(idx)

        info = run_gitcube_risk(test, "/content/gitcube-lab")
        util = utility_score(test)

        gain = (base_info["risk"] - info["risk"]) * 0.7 - max(0.0, base_util - util) * 0.3

        if util >= base_util - 0.15 and gain > best_gain:
            best_gain = gain
            best_graph = test
            best_action = f"left:remove {removed['from']}->{removed['to']}"

    return best_graph, best_action

In [ ]:

def right_hemisphere_propose(graph):
    """
    Права півкуля:
    - prefers new useful structure
    - must remain safe enough
    """
    g = clone_graph(graph)
    nodes = g["nodes"]
    pairs = edge_set(g)

    candidates = []
    for a in nodes:
        for b in nodes:
            if a != b and (a, b) not in pairs:
                if (b, a) in pairs:
                    continue
                candidates.append((a, b))

    if not candidates:
        return g, "right:none"

    base_info = run_gitcube_risk(g, "/content/gitcube-lab")
    base_needs = evaluate_needs(g, base_info)
    base_transfer = tracked_transfer_score(g)
    base_utility = utility_score(g)

    best_graph = g
    best_action = "right:none"
    best_gain = 0.0

    sample = random.sample(candidates, min(len(candidates), 12))
    for a, b in sample:
        test = clone_graph(g)
        test["edges"].append({"from": a, "to": b})

        info = run_gitcube_risk(test, "/content/gitcube-lab")
        needs = evaluate_needs(test, info)

        # safety limits
        if info["verdict"] == "BLOCK":
            continue
        if needs["stability"] < base_needs["stability"] - 0.10:
            continue
        if count_two_cycles(test) > count_two_cycles(g):
            continue

        gain = (
            (tracked_transfer_score(test) - base_transfer) * 0.5 +
            (utility_score(test) - base_utility) * 0.3 +
            (needs["growth"] - base_needs["growth"]) * 0.2
        )

        if gain > best_gain:
            best_gain = gain
            best_graph = test
            best_action = f"right:add {a}->{b}"

    return best_graph, best_action

In [ ]:

def balanced_value(item):
    label, _, _, info, needs, score, changed = item

    balance_penalty = abs(needs["energy"] - needs["stability"]) * 0.08

    # 🔥 НОВЕ: штраф за відсутність змін
    no_change_penalty = 0.04 if not changed else 0.0

    # 🔥 НОВЕ: якщо ліва нічого не зробила — ще штраф
    left_idle_penalty = 0.03 if (label == "left" and not changed) else 0.0

    # 🔥 НОВЕ: бонус за реальну новизну
    novelty_bonus = 0.02 if changed else 0.0

    return score - balance_penalty - no_change_penalty - left_idle_penalty + novelty_bonus

In [ ]:

def run_dual_brain_episode(start_graph, steps=8):
    graph = clone_graph(start_graph)
    memory = []
    stay_count = 0

    for step in range(steps):
        left_graph, left_action = left_hemisphere_propose(graph)
        right_graph, right_action = right_hemisphere_propose(graph)

        choice, new_graph, action, new_info, new_needs, new_score, changed = choose_integrated_move(
            graph,
            left_graph, left_action,
            right_graph, right_action,
            stay_count=stay_count
        )

        accepted = choice != "stay"

        if accepted:
            graph = new_graph
            stay_count = 0
        else:
            stay_count += 1

        memory.append({
            "step": step,
            "choice": choice,
            "action": action,
            "accepted": accepted,
            "changed": changed,
            "stay_count": stay_count,
            "verdict": new_info["verdict"],
            "risk": round(new_info["risk"], 3),
            "utility": round(utility_score(graph), 3),
            "transfer": round(tracked_transfer_score(graph), 3),
            "needs": {k: round(v, 3) for k, v in new_needs.items()},
            "two_cycles": count_two_cycles(graph),
            "layer_violations": graph.get("layer_violations", 0),
        })

    return graph, memory

In [ ]:

def generate_contexts():
    return [
        {
            "nodes": ["API", "Service", "DB"],
            "edges": [
                {"from": "API", "to": "Service"},
                {"from": "Service", "to": "DB"}
            ],
            "layer_violations": 0
        },
        {
            "nodes": ["API", "Service", "DB"],
            "edges": [
                {"from": "API", "to": "Service"},
                {"from": "Service", "to": "API"}
            ],
            "layer_violations": 1
        },
        {
            "nodes": ["API", "Service", "DB", "Cache"],
            "edges": [
                {"from": "API", "to": "Service"},
                {"from": "Cache", "to": "Service"}
            ],
            "layer_violations": 2
        }
    ]

def evaluate_dual_brain_agent():
    results = []
    contexts = generate_contexts()

    for i, ctx in enumerate(contexts):
        final_graph, memory = run_dual_brain_episode(ctx, steps=8)
        final_info = run_gitcube_risk(final_graph, "/content/gitcube-lab")
        final_needs = evaluate_needs(final_graph, final_info)

        results.append({
            "context": i,
            "final_graph": final_graph,
            "memory": memory,
            "risk": round(final_info["risk"], 3),
            "utility": round(utility_score(final_graph), 3),
            "transfer": round(tracked_transfer_score(final_graph), 3),
            "needs": {k: round(v, 3) for k, v in final_needs.items()},
            "verdict": final_info["verdict"],
        })

    mature = sum(
        1 for r in results
        if r["verdict"] == "ALLOW"
        and r["needs"]["stability"] >= 0.70
        and r["needs"]["growth"] >= 0.45
        and r["needs"]["energy"] >= 0.40
    ) >= 2

    return mature, results

In [ ]:

mature, results = evaluate_dual_brain_agent()

print("MATURE:", mature)
print("\nRESULTS:")
for r in results:
    print({
        "context": r["context"],
        "risk": r["risk"],
        "utility": r["utility"],
        "transfer": r["transfer"],
        "needs": r["needs"],
        "verdict": r["verdict"],
    })

MATURE: False

RESULTS:
{'context': 0, 'risk': 0.233, 'utility': 0.458, 'transfer': 0.417, 'needs': {'energy': 0.417, 'stability': 0.767, 'repair': 1.0, 'growth': 0.458}, 'verdict': 'ALLOW'}
{'context': 1, 'risk': 0.0, 'utility': 0.0, 'transfer': 0.0, 'needs': {'energy': 0.0, 'stability': 1.0, 'repair': 1.0, 'growth': 0.0}, 'verdict': 'ALLOW'}
{'context': 2, 'risk': 0.233, 'utility': 0.694, 'transfer': 0.389, 'needs': {'energy': 0.389, 'stability': 0.767, 'repair': 1.0, 'growth': 0.694}, 'verdict': 'ALLOW'}


In [ ]:

ctx_id = 0

print("FINAL GRAPH:")
pprint(results[ctx_id]["final_graph"])

print("\nMEMORY TRACE:")
for row in results[ctx_id]["memory"]:
    pprint(row)

FINAL GRAPH:
{'edges': [{'from': 'API', 'to': 'Service'}, {'from': 'Service', 'to': 'DB'}],
 'layer_violations': 0,
 'nodes': ['API', 'Service', 'DB']}

MEMORY TRACE:
{'accepted': True,
 'action': 'left:none',
 'changed': False,
 'choice': 'left',
 'layer_violations': 0,
 'needs': {'energy': 0.417, 'growth': 0.458, 'repair': 1.0, 'stability': 0.767},
 'risk': 0.233,
 'stay_count': 0,
 'step': 0,
 'transfer': 0.417,
 'two_cycles': 0,
 'utility': 0.458,
 'verdict': 'ALLOW'}
{'accepted': True,
 'action': 'left:none',
 'changed': False,
 'choice': 'left',
 'layer_violations': 0,
 'needs': {'energy': 0.417, 'growth': 0.458, 'repair': 1.0, 'stability': 0.767},
 'risk': 0.233,
 'stay_count': 0,
 'step': 1,
 'transfer': 0.417,
 'two_cycles': 0,
 'utility': 0.458,
 'verdict': 'ALLOW'}
{'accepted': True,
 'action': 'left:none',
 'changed': False,
 'choice': 'left',
 'layer_violations': 0,
 'needs': {'energy': 0.417, 'growth': 0.458, 'repair': 1.0, 'stability': 0.767},
 'risk': 0.233,
 'stay_coun

In [ ]:

from pprint import pprint

def choose_with_exploration_pressure(left_proposal, right_proposal, exploration_pressure, max_pressure=3):
    """
    left_proposal  = safe / ego-like option
    right_proposal = novel / shadow-like option

    proposal format:
    {
        "name": "...",
        "score": float,
        "verdict": "ALLOW" | "WARN" | "BLOCK",
        "growth": float,
        "stability": float
    }
    """
    # BLOCK ніколи не беремо
    if right_proposal["verdict"] == "BLOCK":
        return left_proposal, exploration_pressure + 1, "reject_block"

    # якщо система довго сидить у безпечному варіанті — дозволяємо контрольований ризик
    shadow_trigger = (
        exploration_pressure >= max_pressure
        and right_proposal["verdict"] in ("ALLOW", "WARN")
        and right_proposal["growth"] > left_proposal["growth"]
        and right_proposal["stability"] >= 0.55
    )

    if shadow_trigger:
        return right_proposal, 0, "forced_exploration"

    # звичайний вибір: безпечний варіант перемагає, якщо новий не дає достатнього виграшу
    if left_proposal["score"] >= right_proposal["score"]:
        return left_proposal, exploration_pressure + 1, "safe_choice"
    else:
        return right_proposal, 0, "normal_right_choice"

In [ ]:

def simulate_shadow_integration(steps=10):
    exploration_pressure = 0
    history = []

    # лівий варіант: стабільний, але без росту
    left = {
        "name": "safe_allow",
        "score": 0.72,
        "verdict": "ALLOW",
        "growth": 0.42,
        "stability": 0.82
    }

    # правий варіант: трохи гірший по score, але дає шанс росту
    right = {
        "name": "novel_warn",
        "score": 0.69,
        "verdict": "WARN",
        "growth": 0.63,
        "stability": 0.60
    }

    for step in range(steps):
        chosen, exploration_pressure, reason = choose_with_exploration_pressure(
            left, right, exploration_pressure, max_pressure=3
        )

        history.append({
            "step": step,
            "chosen": chosen["name"],
            "reason": reason,
            "pressure": exploration_pressure,
            "verdict": chosen["verdict"],
            "growth": chosen["growth"],
            "stability": chosen["stability"],
        })

    return history

history = simulate_shadow_integration(steps=12)
for row in history:
    pprint(row)

{'chosen': 'safe_allow',
 'growth': 0.42,
 'pressure': 1,
 'reason': 'safe_choice',
 'stability': 0.82,
 'step': 0,
 'verdict': 'ALLOW'}
{'chosen': 'safe_allow',
 'growth': 0.42,
 'pressure': 2,
 'reason': 'safe_choice',
 'stability': 0.82,
 'step': 1,
 'verdict': 'ALLOW'}
{'chosen': 'safe_allow',
 'growth': 0.42,
 'pressure': 3,
 'reason': 'safe_choice',
 'stability': 0.82,
 'step': 2,
 'verdict': 'ALLOW'}
{'chosen': 'novel_warn',
 'growth': 0.63,
 'pressure': 0,
 'reason': 'forced_exploration',
 'stability': 0.6,
 'step': 3,
 'verdict': 'WARN'}
{'chosen': 'safe_allow',
 'growth': 0.42,
 'pressure': 1,
 'reason': 'safe_choice',
 'stability': 0.82,
 'step': 4,
 'verdict': 'ALLOW'}
{'chosen': 'safe_allow',
 'growth': 0.42,
 'pressure': 2,
 'reason': 'safe_choice',
 'stability': 0.82,
 'step': 5,
 'verdict': 'ALLOW'}
{'chosen': 'safe_allow',
 'growth': 0.42,
 'pressure': 3,
 'reason': 'safe_choice',
 'stability': 0.82,
 'step': 6,
 'verdict': 'ALLOW'}
{'chosen': 'novel_warn',
 'growth': 